# CropCop Track B — EAAI External Validation

Final paper-first Track-B execution. It reuses the completed Notebook-00 materialization, performs exact-content leakage screening, and evaluates frozen R07 S1/S2/S3 directly in the native 120-class space.

**Kaggle:** attach exactly the existing infrastructure + external Track-B datasets, enable one GPU, enable Internet. No Kaggle secret is required.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

EXPECTED_SOURCE_COMMIT = '7d9c09dfc6ec6bad640ee8af429abd1f6ab688d1'
REPO = Path('/kaggle/working/ResearchWork-CropCop-v7')
OUTPUT = Path('/kaggle/working/trackb_eaai_results')
INPUT = Path('/kaggle/input')

if OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise RuntimeError(f'Refusing to overwrite non-empty output: {OUTPUT}')
free_gib = shutil.disk_usage('/kaggle/working').free / (1024**3)
if free_gib < 5:
    raise RuntimeError(f'Insufficient /kaggle/working free space: {free_gib:.2f} GiB')

manifests = sorted(INPUT.glob('**/TRACKB_INPUT_MANIFEST.json'))
roles=[]
for p in manifests:
    roles.append(str(json.loads(p.read_text(encoding='utf-8')).get('role','')))
expected_roles={'core','historical_compare','gvlid_v5','irish_potato'}
if set(roles)!=expected_roles or len(roles)!=4:
    raise RuntimeError(f'Attach exactly one of each Track-B role; observed={roles}')
infra=list(INPUT.glob('**/TRACKB_INFRASTRUCTURE_BUNDLE.json'))
external=list(INPUT.glob('**/TRACKB_EXTERNAL_BUNDLE.json'))
if len(infra)!=1 or len(external)!=1:
    raise RuntimeError('Attach exactly one infrastructure and one external Notebook-00 bundle')
iid=str(json.loads(infra[0].read_text())['materialization_id'])
eid=str(json.loads(external[0].read_text())['materialization_id'])
if iid != eid or iid != '98ff32c3c7ce2e66ffe52be97932a8b25f3ce50314c844abdd33077c99d90d78':
    raise RuntimeError(f'Wrong/mixed Notebook-00 materialization: infra={iid}, external={eid}')

gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True,timeout=60)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No Kaggle GPU detected; enable a GPU accelerator before running')
print({'status':'PASS_KAGGLE_PREFLIGHT','roles':sorted(roles),'materialization_id':iid,'free_working_gib':round(free_gib,2),'gpu':gpu.stdout.strip().splitlines()})


In [ ]:
def run_retry(cmd, *, attempts=3, timeout=300, cwd=None):
    last_message=''
    for attempt in range(1, attempts+1):
        try:
            p=subprocess.run(cmd,cwd=cwd,capture_output=True,text=True,timeout=timeout)
            if p.returncode==0:
                if p.stdout.strip(): print(p.stdout[-3000:])
                return p
            last_message=(p.stderr or p.stdout or '')[-4000:]
        except subprocess.TimeoutExpired as exc:
            stdout=exc.stdout.decode(errors='replace') if isinstance(exc.stdout,bytes) else (exc.stdout or '')
            stderr=exc.stderr.decode(errors='replace') if isinstance(exc.stderr,bytes) else (exc.stderr or '')
            last_message=(stderr or stdout or f'timed out after {timeout}s')[-4000:]
        print(f'attempt {attempt}/{attempts} failed:', last_message[-2000:])
        if attempt < attempts: time.sleep(5*attempt)
    raise RuntimeError(f'command failed after {attempts} attempts: {cmd}\n{last_message}')

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','init',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'remote','add','origin','https://github.com/rana-m-ahmed/ResearchWork-CropCop.git'],check=True)
run_retry(['git','-C',str(REPO),'fetch','--depth','1','origin',EXPECTED_SOURCE_COMMIT],attempts=3,timeout=180)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
observed=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if observed != EXPECTED_SOURCE_COMMIT: raise RuntimeError(f'Source commit mismatch: {observed}')
print({'status':'PASS_PINNED_SOURCE','commit':observed})


In [ ]:
import importlib.metadata as md
required={'torch':'2.12.1','torchvision':'0.27.1','numpy':'2.5.2','Pillow':'12.3.0'}
drift=[]
before={}
for package,version in required.items():
    try: current=md.version(package)
    except md.PackageNotFoundError: current=None
    before[package]=current
    if (current or '').split('+',1)[0] != version: drift.append(f'{package}=={version}')
print({'runtime_before':before})
if drift:
    run_retry([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-cache-dir','--upgrade',*drift],attempts=2,timeout=1200)

probe_code="""
import json, torch, torchvision, numpy, PIL
versions={'torch':torch.__version__.split('+',1)[0],'torchvision':torchvision.__version__.split('+',1)[0],'numpy':numpy.__version__,'Pillow':PIL.__version__}
expected={'torch':'2.12.1','torchvision':'0.27.1','numpy':'2.5.2','Pillow':'12.3.0'}
assert versions == expected, (versions, expected)
assert torch.cuda.is_available(), 'CUDA unavailable after runtime repair'
print(json.dumps({'status':'PASS_FRESH_RUNTIME','versions':versions,'cuda_devices':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))
"""
probe=subprocess.run([sys.executable,'-c',probe_code],capture_output=True,text=True,timeout=180)
if probe.returncode!=0: raise RuntimeError('Fresh runtime verification failed:\n'+(probe.stderr or probe.stdout)[-4000:])
print(probe.stdout.strip())


In [ ]:
runner=REPO/'journal_extension/scripts/run_trackb_eaai_external_validation.py'
protocol=REPO/'journal_extension/track_b_r07/TRACKB_EAAI_PROTOCOL_v2.json'
smoke="""
import sys, json
sys.path.insert(0, '/kaggle/working/ResearchWork-CropCop-v7/journal_extension/src')
from cropcop_je.trackb_eaai_common import load_protocol
p=load_protocol('/kaggle/working/ResearchWork-CropCop-v7/journal_extension/track_b_r07/TRACKB_EAAI_PROTOCOL_v2.json')
print(json.dumps({'status':'PASS_V7_IMPORT_SMOKE','protocol_id':p['protocol_id'],'native_output_classes':p['prediction_policy']['native_output_classes']}))
"""
p=subprocess.run([sys.executable,'-c',smoke],capture_output=True,text=True,timeout=120)
if p.returncode!=0: raise RuntimeError('v7 import/protocol smoke failed:\n'+(p.stderr or p.stdout)[-4000:])
print(p.stdout.strip())

cmd=[sys.executable,'-B',str(runner),'--input-root','/kaggle/input','--output-root',str(OUTPUT),'--protocol',str(protocol),'--mode','all','--device','cuda:0','--batch-size','64','--audit-workers','8','--loader-workers','4']
print('Launching final Track-B EAAI run:', ' '.join(cmd), flush=True)
subprocess.run(cmd,cwd=REPO,check=True,env={**os.environ,'PYTHONDONTWRITEBYTECODE':'1'})


In [ ]:
final_path=OUTPUT/'TRACKB_FINAL_MANIFEST.json'
if not final_path.is_file(): raise RuntimeError('Final Track-B manifest missing')
final=json.loads(final_path.read_text(encoding='utf-8'))
if final.get('status')!='PASS_TRACKB_EAAI_EXTERNAL_VALIDATION': raise RuntimeError(f"Unexpected terminal status: {final.get('status')}")
if final.get('v1_test_accessed') is not False: raise RuntimeError('V1 test closure was not preserved')
archive=Path(shutil.make_archive('/kaggle/working/CropCop_TrackB_EAAI_External_Validation','zip',root_dir=OUTPUT))
print(json.dumps({'status':final['status'],'external_prediction_count':final['external_prediction_count'],'manifest_sha256':final['manifest_sha256'],'results_dir':str(OUTPUT),'evidence_zip':str(archive)},indent=2,sort_keys=True))
